# TCGA-BRCA TE expression by PAM50 subtype

This notebook is a public, reproducible quick look at transposable-element expression in TCGA-BRCA without reprocessing FASTQ/BAM files. It uses the precomputed REdiscoverTEdata intergenic TE logCPM matrix and joins it to UCSC Xena TCGA-BRCA PAM50 labels.

Public sources:

- REdiscoverTE paper: https://www.nature.com/articles/s41467-019-13035-2
- REdiscoverTEdata package: http://research-pub.gene.com/REdiscoverTEpaper/data/REdiscoverTEdata_1.0.1.tar.gz
- UCSC Xena TCGA-BRCA clinical matrix: https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/BRCA_clinicalMatrix

The notebook downloads data into `_local/cache/brca_te/`, which is ignored by git. It requires Python plus R. If R's `Biobase` package is missing, the notebook installs `BiocManager` and `Biobase` into `_local/cache/brca_te/r-lib/`.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm

ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent

CACHE = ROOT / "_local" / "cache" / "brca_te"
OUT = CACHE / "outputs"
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

REDISCOVERTE_URL = "http://research-pub.gene.com/REdiscoverTEpaper/data/REdiscoverTEdata_1.0.1.tar.gz"
CLINICAL_URL = "https://tcga.xenahubs.net/download/TCGA.BRCA.sampleMap/BRCA_clinicalMatrix"
TARBALL = CACHE / "REdiscoverTEdata_1.0.1.tar.gz"
RDS = CACHE / "REdiscoverTEdata" / "inst" / "Fig4_data" / "eset_TCGA_TE_intergenic_logCPM.RDS"
CLINICAL = CACHE / "BRCA_clinicalMatrix.tsv"
EXPECTED_SHA256 = {
    "rediscovertre_tarball": "a4dda87737fb95c170552c50c85e7a8f0c2eb1d2ab25e0816fc5376e0fd32d04",
    "rediscovertre_rds": "c34775400a85d425936dd303986b67eb071422b648aa5d64b70cd7e5cb71d4b3",
    "xena_clinical_matrix": "822db81a4f345a455bf9d26bea59423761828ef60bfc2213e335e6351bfc6e1c",
}

print(f"Repo root: {ROOT}")
print(f"Cache: {CACHE}")

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_sha256(path: Path, expected: str) -> None:
    actual = sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"Checksum mismatch for {path}: expected {expected}, got {actual}")
    print(f"Verified sha256: {path.name} {actual}")

def download(url: str, path: Path, expected_sha256: str) -> None:
    if path.exists() and path.stat().st_size > 0:
        if sha256_file(path) == expected_sha256:
            print(f"Already present and verified: {path.name} ({path.stat().st_size:,} bytes)")
            return
        print(f"Existing file failed checksum; re-downloading {path.name}")
        path.unlink()
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path.name} ({path.stat().st_size:,} bytes)")
    verify_sha256(path, expected_sha256)

download(REDISCOVERTE_URL, TARBALL, EXPECTED_SHA256["rediscovertre_tarball"])
download(CLINICAL_URL, CLINICAL, EXPECTED_SHA256["xena_clinical_matrix"])

if RDS.exists() and sha256_file(RDS) != EXPECTED_SHA256["rediscovertre_rds"]:
    print("Existing RDS failed checksum; re-extracting from verified tarball")
    RDS.unlink()
if not RDS.exists():
    wanted = "REdiscoverTEdata/inst/Fig4_data/eset_TCGA_TE_intergenic_logCPM.RDS"
    with tarfile.open(TARBALL, "r:gz") as tf:
        tf.extract(wanted, CACHE)
    print(f"Extracted: {RDS} ({RDS.stat().st_size:,} bytes)")
else:
    print(f"RDS already present: {RDS} ({RDS.stat().st_size:,} bytes)")
verify_sha256(RDS, EXPECTED_SHA256["rediscovertre_rds"])

In [ ]:
if not shutil.which("Rscript"):
    raise RuntimeError("Rscript is required to read the Biobase ExpressionSet RDS.")

R_LIB = CACHE / "r-lib"
R_LIB.mkdir(parents=True, exist_ok=True)

def r_string(path: Path) -> str:
    return json.dumps(str(path))

r_code = f'''
lib <- {r_string(R_LIB)}
.libPaths(c(lib, .libPaths()))
if (!requireNamespace('BiocManager', quietly=TRUE)) {{
  install.packages('BiocManager', repos='https://cloud.r-project.org', lib=lib)
}}
if (!requireNamespace('Biobase', quietly=TRUE)) {{
  BiocManager::install('Biobase', lib=lib, ask=FALSE, update=FALSE)
}}
suppressPackageStartupMessages(library(Biobase))

out_dir <- {r_string(OUT)}
eset <- readRDS({r_string(RDS)})
expr <- exprs(eset)
pd <- pData(eset)
fd <- fData(eset)

keep <- pd$indication == 'BRCA' & substr(colnames(expr), 14, 15) == '01'
expr_b <- expr[, keep, drop=FALSE]
sample_barcode <- colnames(expr_b)
sample_id <- substr(sample_barcode, 1, 15)

clinical <- read.delim({r_string(CLINICAL)}, check.names=FALSE, stringsAsFactors=FALSE)
clinical$key <- clinical$sampleID
clin_idx <- match(sample_id, clinical$key)
subtype <- clinical$PAM50Call_RNAseq[clin_idx]
subtype[subtype == ''] <- NA

rep_class <- sub('\\\\?$', '', as.character(fd$repClass))
class_order <- c('LTR', 'DNA', 'LINE', 'SINE', 'Satellite', 'Retroposon')
rows <- list()
for (cl in class_order) {{
  idx <- which(rep_class == cl)
  vals <- apply(expr_b[idx,,drop=FALSE], 2, median, na.rm=TRUE)
  rows[[cl]] <- data.frame(sample=sample_barcode, sampleID=sample_id, subtype=subtype, te_class=cl, median_logCPM=as.numeric(vals), feature_count=length(idx), stringsAsFactors=FALSE)
}}
summary_df <- do.call(rbind, rows)
summary_df <- summary_df[!is.na(summary_df$subtype),]
write.csv(summary_df, file.path(out_dir, 'brca_te_core_class_by_pam50_summary.csv'), row.names=FALSE)

meta <- data.frame(metric=c('te_matrix_features','te_matrix_samples','brca_primary_tumors','clinical_matched','pam50_labeled_primary_tumors','classes_plotted'), value=c(nrow(expr), ncol(expr), ncol(expr_b), sum(!is.na(clin_idx)), length(unique(summary_df$sample)), paste(class_order, collapse=';')), stringsAsFactors=FALSE)
write.csv(meta, file.path(out_dir, 'brca_te_core_class_by_pam50_metadata.csv'), row.names=FALSE)
print(meta)
'''

script = CACHE / "convert_rediscovertre_brca_te.R"
script.write_text(r_code)
env = os.environ.copy()
env["R_LIBS_USER"] = str(R_LIB)
subprocess.run(["Rscript", str(script)], check=True, env=env)

summary_csv = OUT / "brca_te_core_class_by_pam50_summary.csv"
metadata_csv = OUT / "brca_te_core_class_by_pam50_metadata.csv"
print(summary_csv)
print(metadata_csv)

In [ ]:
summary = pd.read_csv(OUT / "brca_te_core_class_by_pam50_summary.csv")
metadata = pd.read_csv(OUT / "brca_te_core_class_by_pam50_metadata.csv")

summary["subtype_label"] = summary["subtype"].replace({"Normal": "Normal-like"})
subtypes = ["Basal", "Her2", "LumA", "LumB", "Normal-like"]
classes = ["LTR", "DNA", "LINE", "SINE", "Satellite", "Retroposon"]
summary["subtype_label"] = pd.Categorical(summary["subtype_label"], categories=subtypes, ordered=True)
summary["te_class"] = pd.Categorical(summary["te_class"], categories=classes, ordered=True)

heat = (
    summary.groupby(["subtype_label", "te_class"], observed=True)["median_logCPM"]
    .median()
    .unstack("te_class")
    .loc[subtypes, classes]
)

display(metadata)
display(heat.round(3))

In [ ]:
fig = plt.figure(figsize=(13.7, 8.4), constrained_layout=True)
gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 2.25])

ax = fig.add_subplot(gs[0])
vals = heat.to_numpy(dtype=float)
norm = TwoSlopeNorm(vcenter=0, vmin=np.nanmin(vals), vmax=np.nanmax(vals))
im = ax.imshow(vals, aspect="auto", cmap="RdBu_r", norm=norm)
ax.set_xticks(range(len(classes)), labels=classes, fontsize=11)
ax.set_yticks(range(len(subtypes)), labels=subtypes, fontsize=11)
ax.set_title("Subtype medians from per-sample TE-class median logCPM", fontsize=14, weight="bold", pad=10)
for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        ax.text(j, i, f"{vals[i, j]:.2f}", ha="center", va="center", fontsize=10, color="black")
cb = fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.8, pad=0.02)
cb.set_label("Median logCPM", fontsize=10)

subgs = gs[1].subgridspec(2, 3, wspace=0.24, hspace=0.30)
colors = {"Basal":"#4E79A7", "Her2":"#E15759", "LumA":"#59A14F", "LumB":"#F28E2B", "Normal-like":"#B07AA1"}
for k, cl in enumerate(classes):
    axv = fig.add_subplot(subgs[k // 3, k % 3])
    data = [summary[(summary["te_class"] == cl) & (summary["subtype_label"] == st)]["median_logCPM"].dropna().to_numpy() for st in subtypes]
    parts = axv.violinplot(data, positions=np.arange(len(subtypes)), widths=0.8, showmeans=False, showextrema=False, showmedians=False)
    for body, st in zip(parts["bodies"], subtypes):
        body.set_facecolor(colors[st])
        body.set_edgecolor("#333333")
        body.set_alpha(0.62)
        body.set_linewidth(0.6)
    meds = np.array([np.median(x) if len(x) else np.nan for x in data])
    q1 = np.array([np.percentile(x, 25) if len(x) else np.nan for x in data])
    q3 = np.array([np.percentile(x, 75) if len(x) else np.nan for x in data])
    axv.scatter(np.arange(len(subtypes)), meds, color="black", s=16, zorder=3)
    axv.vlines(np.arange(len(subtypes)), q1, q3, color="black", linewidth=1.4, zorder=2)
    axv.axhline(0, color="#999999", lw=0.7, zorder=0)
    feature_count = int(summary.loc[summary.te_class == cl, "feature_count"].iloc[0])
    axv.set_title(f"{cl} ({feature_count} subfamilies/features)", fontsize=11, weight="bold")
    axv.set_xticks(range(len(subtypes)), labels=subtypes, rotation=30, ha="right", fontsize=9)
    axv.tick_params(axis="y", labelsize=9)
    if k % 3 == 0:
        axv.set_ylabel("Per-sample median logCPM", fontsize=10, labelpad=12)

fig.suptitle(
    "TCGA-BRCA transposable-element expression by PAM50 subtype\n"
    "REdiscoverTE intergenic TE matrix joined to UCSC Xena PAM50Call_RNAseq labels",
    fontsize=16,
    weight="bold",
)

plot_path = OUT / "brca_te_core_class_by_pam50_heatmap_violin.png"
release_plot_path = ROOT / "docs" / "assets" / "brca-te-subtype-heatmap-violin.png"
release_plot_path.parent.mkdir(parents=True, exist_ok=True)
for target in [plot_path, release_plot_path]:
    fig.savefig(target, dpi=180, bbox_inches="tight", pad_inches=0.18)
    print(target)
plt.show()

In [ ]:
counts = (
    summary[summary["te_class"] == classes[0]]["subtype_label"]
    .value_counts()
    .reindex(subtypes)
    .fillna(0)
    .astype(int)
    .to_dict()
)
class_ranges = {cl: float(heat[cl].max() - heat[cl].min()) for cl in classes}
interpretation = {
    "samples_by_subtype": counts,
    "largest_subtype_median_range": sorted(class_ranges.items(), key=lambda item: item[1], reverse=True),
    "heatmap_medians": heat.round(3).to_dict(),
    "plain_language": (
        "This class-level quick look shows the clearest PAM50 separation in Retroposon, "
        "with Basal tumors higher than other PAM50 groups. Satellite and DNA show smaller shifts; "
        "LINE, SINE, and LTR are comparatively stable at the class-median level. "
        "Treat this as orientation, not subfamily-level differential expression."
    ),
}

interpretation_path = OUT / "brca_te_core_class_by_pam50_interpretation.json"
interpretation_path.write_text(json.dumps(interpretation, indent=2))
print(json.dumps(interpretation, indent=2))